# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriKeerthi16-b/Machine-Learning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
!git clone https://github.com/SriKeerthi16-b/Machine-Learning.git

Cloning into 'Machine-Learning'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 166 (delta 70), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 1.89 MiB | 9.52 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [5]:
import pandas as pd
import numpy as np

DATA_PATH = "Machine-Learning/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Rows: 30000
Columns: 45
Declining rate: 0.542


In [6]:
import pandas as pd
import numpy as np

# Load the FlyRank dataset
DATA_PATH = "Machine-Learning/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Create the target label
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Rows: 30000
Columns: 45
Declining rate: 0.542


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the two methodology questions against the paper's reported setup

paper_checks = {
    "Finding 1 label": "Recent traffic trend direction (growing vs declining)",
    "Finding 1 validation concern": "Observed comparison; does not establish causation",
    "Finding 2 label": "Growing vs declining page outcome",
    "Finding 2 validation design": "Unseen pages / unseen brands"
}

for key, value in paper_checks.items():
    print(f"{key}: {value}")

Finding 1 label: Recent traffic trend direction (growing vs declining)
Finding 1 validation concern: Observed comparison; does not establish causation
Finding 2 label: Growing vs declining page outcome
Finding 2 validation design: Unseen pages / unseen brands


### Finding 1: Content lifecycle and decline

The paper reports that growing pages were younger on average than declining pages: about 185 days versus 228 days. The label comes from recent traffic trend direction, where pages are classified as growing or declining based on their recent performance. The comparison is useful for describing an observed pattern, but the validation design does not prove that page age causes decline. The paper itself treats these findings as patterns rather than causal evidence. A stronger methodology question is whether the same relationship remains when pages are compared within similar age, topic, or client groups.

### Finding 2: Growth prediction on unseen brands

The paper reports that its growth model performs around 90% on new pages from known brands and around 75% on completely unseen brands. The outcome is based on whether pages are classified as growing or declining. This validation design is stronger than a simple random row split because it tests performance on entities the model has not seen. However, the result should still be interpreted as predictive decision-support rather than proof that the model causes growth. A useful methodology question is whether the result remains stable across different groups and time periods.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

model_df = df.dropna(
    subset=feature_cols + ["is_declining_label", "client_id"]
).copy()

X = model_df[feature_cols]
y = model_df["is_declining_label"]
groups = model_df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_scores = model.predict_proba(X_test)[:, 1]

ranked = pd.DataFrame({
    "actual": y_test.values,
    "score": test_scores
}).sort_values("score", ascending=False)

honest_precision_50 = ranked.head(50)["actual"].mean()

print("Honest grouped Precision@50:",
      round(honest_precision_50, 3))

Honest grouped Precision@50: 0.64


My Week-5 model was a Decision Tree evaluated with Precision@50. The earlier result was measured before applying the final client-grouped holdout design. I re-run the same model using a grouped split so that pages from the same client cannot appear in both training and test data. The comparison shows how much the measured result changes when the evaluation better matches the intended use on unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature leakage audit

target_col = "is_declining_label"
group_col = "client_id"

print("Target column:", target_col)
print("Group column:", group_col)
print("Final features:", feature_cols)

print("\nTarget included in features:",
      target_col in feature_cols)

print("Client ID included in features:",
      group_col in feature_cols)

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Client overlap between train/test:",
      len(train_clients & test_clients))

assert target_col not in feature_cols
assert group_col not in feature_cols
assert len(train_clients & test_clients) == 0

print("\nLeakage audit passed.")

Target column: is_declining_label
Group column: client_id
Final features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Target included in features: False
Client ID included in features: False
Client overlap between train/test: 0

Leakage audit passed.


I audit the final feature set for obvious target leakage. The target is derived from trend direction, so trend-derived or future-performance fields should not be used as predictors. I also check that the target column and client identifier are not included in the model feature list. The grouped split provides an additional client-level leakage check.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Safe claim:")
print(
    "On this dataset, the Decision Tree showed measured "
    "Precision@50 under the tested grouped validation split. "
    "This is an observed and directional result that may support "
    "prioritizing pages for review, but it does not prove that "
    "the model will cause or reliably predict future traffic outcomes."
)

Safe claim:
On this dataset, the Decision Tree showed measured Precision@50 under the tested grouped validation split. This is an observed and directional result that may support prioritizing pages for review, but it does not prove that the model will cause or reliably predict future traffic outcomes.


My earlier claim could have been stated too strongly as: "The model can accurately identify pages that will decline."

A safer rewrite is:

"On this dataset, the Decision Tree showed measured Precision@50 under the tested grouped validation split. This is an observed and directional result that may support prioritizing pages for review, but it does not prove that the model will cause or reliably predict future traffic outcomes."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.